# L4b · Is L4's near-zero result EfficientSAM3 or our setup? (Kaggle T4, about 10 minutes)

Task **L4b**, a check on **L4** (`l4-efficientsam3-building-finding.ipynb`, run 2 on 18 Nov). Built 18 Nov, at Abrar's go. It trains nothing and decides nothing about the system: it says why L4 came out as it did.

**Why**
- In L4, EfficientSAM3's three released models (EV-M, RV-M, TV-M) found almost no buildings on our validation before-photos: Building IoU 0.010–0.054 on the 605 clean pairs, against full SAM 3's 0.6302. 56–94% of photos got no detection at the score threshold 0.5.
- Every checkpoint loaded fully, and SAM 3's reference was reproduced. But L4 never checked that the same setup finds anything in an ordinary photo, so a broken setup and a model that can't see buildings from above would look the same.

**What it does**
- **Check A, the repo's own example:** each model on `sam3/assets/dog_person.jpeg` with the prompts "dog" and "person", as the repo's `efficientsam3_image_predictor_example.py` does, at the score threshold 0.5. Overlays are saved to look at.
- **Check B, 21 satellite photos:** L4's first 20 clean validation pairs (its CPU-timing set) and its densest probe photo, before-photos whole, as L4's `pre_grid1`.
  - Each model returns everything scoring above 0.01. Building IoU, detections and the top score are then counted at the thresholds 0.5 (L4's and E3's), 0.3, 0.1 and 0.05.
  - A score is the query's confidence times the model's "this concept is in the photo" score, so a model that doesn't see buildings at all gives low scores everywhere.
  - The prompt "building" decides. "house" and "roof" are reported beside it, deciding nothing.
  - SAM 3 on the same 21 photos (E3, whole photos, threshold 0.5) is the reference: Building IoU **0.2812**.
  - Overlays of two photos SAM 3 does well on (Moore tornado, Turkey) at 0.05 are saved to look at.
- Same code and weights as L4 run 2: repo `SimonZeng7108/efficientsam3@bd0936c` (fetched exactly), weights `Simon7108528/EfficientSAM3@85b05896`, MobileCLIP-S0 at 16 tokens, half precision on the T4.

**Checks (the notebook stops if one fails)**
1. The 21 photos' true building pixels equal E3's, in total within 0.5%: the same photos and masks.
2. Every checkpoint fills at least 95% of its model's parameters, as in L4.

**How it's read (fixed before the run)**
- **A fails** (a model finds no dog or no person at 0.5): our setup is broken, and L4's result says nothing about EfficientSAM3 until it is fixed.
- **A passes, and with "building" no model reaches Building IoU 0.10 at any threshold** (a third of SAM 3's 0.28 here): the models don't see our buildings, and L4 stands as measured.
- **A passes, and some model reaches 0.20 or more with "building" at some threshold** (about 70% of SAM 3 here): the models find buildings but score them low. A new run on the 605 clean pairs at that model and threshold becomes a candidate, with its own rule.
- **Between 0.10 and 0.20:** partly; reported, nothing decided.

**Expectation (fixed before the run)**
- A passes: the models find the dog and the person.
- With "building", no model reaches 0.10 at any threshold, and most top scores stay under 0.3. "house" and "roof" do no better. L4's result is then about EfficientSAM3 on overhead photos, not about our setup.

**How to run on Kaggle** (any account with GPU quota)
1. Settings: Accelerator **GPU T4 x2** (one is used), Internet **On**. No secret is needed.
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. **Save Version → Save & Run All (Commit).** About 10 minutes, estimated. There's no smoke test: the whole run is small.

**Outputs** in `/kaggle/working/L4b/`: `README.md`, `config.json`, `check_a_example.csv`, `check_b_per_photo.csv`, `check_b_summary.csv`, and `overlays/*.png`.

In [1]:
# ── Cell 1: EfficientSAM3 at L4's commit, imported as in L4 run 2 ──
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

ESAM3_REPO = "https://github.com/SimonZeng7108/efficientsam3.git"
ESAM3_DIR = Path("/tmp/efficientsam3")
ESAM3_PIN = "bd0936c788fed8d51fa799437f05abd97b401b06"      # the commit L4 ran on


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


def git(*args):
    return subprocess.check_output(["git", "-C", str(ESAM3_DIR), *args], text=True).strip()


if not ESAM3_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", ESAM3_REPO, str(ESAM3_DIR)], check=True)
if git("rev-parse", "HEAD") != ESAM3_PIN:
    git("fetch", "--depth", "1", "origin", ESAM3_PIN)
    git("checkout", "--quiet", "FETCH_HEAD")
ESAM3_COMMIT = git("rev-parse", "HEAD")
assert ESAM3_COMMIT == ESAM3_PIN, ESAM3_COMMIT

# As L4 run 2: `pip install -e` isn't seen by a running kernel, so the clone's package folder goes on sys.path
_builders = sorted(ESAM3_DIR.rglob("sam3/model_builder.py"), key=lambda p: len(p.parts))
assert _builders, "sam3/model_builder.py not found in the clone"
ESAM3_PKG_ROOT = _builders[0].parent.parent
if str(ESAM3_PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(ESAM3_PKG_ROOT))
for _m in [m for m in sys.modules if m == "sam3" or m.startswith("sam3.")]:
    del sys.modules[_m]
importlib.invalidate_caches()
pip("timm>=1.0.17", "ftfy==6.1.1", "regex", "iopath", "einops", "pycocotools", "psutil",
    "open_clip_torch", "huggingface_hub", "safetensors")
if importlib.util.find_spec("pkg_resources") is None:
    pip("setuptools<81")

import torch
import sam3
import sam3.model_builder as mb

assert Path(sam3.__file__).resolve().is_relative_to(ESAM3_DIR.resolve()), f"wrong sam3 imported: {sam3.__file__}"
assert hasattr(mb, "build_efficientsam3_image_model")
print(f"✓ EfficientSAM3 at {ESAM3_COMMIT[:7]}; torch {torch.__version__}, python {sys.version.split()[0]}")
print(f"  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (needs GPU T4)'}")

Cloning into '/tmp/efficientsam3'...
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✓ EfficientSAM3 at bd0936c; torch 2.11.0+cu130, python 3.13.16
  GPU: Tesla T4


In [2]:
# ── Cell 2: Settings, the three models, the 21 photos and SAM 3's results on them (E3) ──
import json
import time

import cv2
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_efficientsam3_image_model

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/L4b")
(OUT_DIR / "overlays").mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "L4b runs on a GPU, in half precision as L4 did"
assert MANIFEST_PATH.exists(), "attach abrarmohammedtanzim/disasterm3-mirror"

HF_REPO, HF_SHA = "Simon7108528/EfficientSAM3", "85b05896928f974e308f889d7ccb2eefc069de98"   # L4's weights
TEXT_ENCODER, TEXT_CONTEXT = "MobileCLIP-S0", 16
STUDENTS = [
    {"label": "EV-M", "rfilename": "efficientsam3_ft/efficientsam3_efficientvit.pt", "backbone_type": "efficientvit", "model_name": "b1"},
    {"label": "RV-M", "rfilename": "efficientsam3_ft/efficientsam3_repvit.pt", "backbone_type": "repvit", "model_name": "m1.1"},
    {"label": "TV-M", "rfilename": "efficientsam3_ft/efficientsam3_tinyvit.pt", "backbone_type": "tinyvit", "model_name": "11m"},
]
FLOOR = 0.01                                   # the processor keeps every detection above this
THRESHOLDS = (0.5, 0.3, 0.1, 0.05)             # 0.5 is L4's and E3's; the others are counted after
MIN_INSTANCE_PX = 10                           # as E3 and L4
PROMPTS = ("building", "house", "roof")        # "building" decides; the other two are reported beside it
EXAMPLE = ("dog_person.jpeg", ("dog", "person"))   # the repo's own example photo and prompts
OVERLAY_PHOTOS = ("moore_tornado_00000106_post_disaster.png", "turkey_earthquake5_post_884.png")

# The 21 photos: L4's first 20 clean validation pairs in split order (its CPU-timing set) and its densest probe photo.
# SAM 3 on their before-photos, whole (pre_grid1), from E3's sam3_s1.csv:
# (image, intersection px, union px, instances, true buildings, true building px)
SAM3_REF = [
    ("turkey_earthquake3_post_692.png", 70343, 585315, 13, 38, 583832),
    ("hurricane_florence_00000315_post_disaster.png", 1237, 1725, 2, 3, 1407),
    ("nepal_flooding_00000278_post_disaster.png", 0, 1015, 0, 6, 1015),
    ("santa_rosa_wildfire_00000201_post_disaster.png", 2059, 4087, 3, 5, 3916),
    ("hurricane_michael_00000224_post_disaster.png", 22435, 40038, 23, 30, 34032),
    ("hurricane_matthew_00000217_post_disaster.png", 2629, 4092, 11, 12, 3205),
    ("turkey_earthquake5_post_884.png", 195826, 301413, 85, 96, 286572),
    ("hurricane_michael_00000530_post_disaster.png", 15228, 16549, 1, 1, 16423),
    ("nepal_flooding_00000559_post_disaster.png", 0, 744, 0, 1, 744),
    ("pinery_bushfire_00001611_post_disaster.png", 1758, 2451, 1, 3, 2329),
    ("midwest_flooding_00000336_post_disaster.png", 23168, 35773, 26, 32, 28506),
    ("socal_fire_00001000_post_disaster.png", 40203, 77584, 37, 54, 72545),
    ("turkey_earthquake3_post_654.png", 225176, 567773, 25, 35, 557630),
    ("kalehe_flooding_post_1010.png", 0, 8742, 8, 22, 5884),
    ("midwest_flooding_00000031_post_disaster.png", 34956, 276671, 20, 174, 272173),
    ("nepal_flooding_00000008_post_disaster.png", 11401, 28763, 24, 35, 27083),
    ("turkey_earthquake5_post_920.png", 112029, 392482, 34, 138, 385286),
    ("portugal_wildfire_00000898_post_disaster.png", 3851, 5823, 4, 5, 4912),
    ("hurricane_florence_00000039_post_disaster.png", 0, 1373, 0, 4, 1373),
    ("moore_tornado_00000106_post_disaster.png", 155741, 194887, 80, 85, 165674),
    ("turkey_earthquake0_post_456.png", 7105, 742413, 2, 34, 741855),
]
SAM3_DF = pd.DataFrame(SAM3_REF, columns=["image", "inter", "union", "instances", "gt_blobs", "gt_px"])
SAM3_IOU = SAM3_DF["inter"].sum() / SAM3_DF["union"].sum()
assert len(SAM3_DF) == 21 and round(SAM3_IOU, 4) == 0.2812


def checkpoint_coverage(model, ckpt):
    """Share of the model's parameter values whose names the checkpoint holds (L4 run 2's check)."""
    sd = torch.load(ckpt, map_location="cpu", weights_only=True)
    sd = sd["model"] if isinstance(sd.get("model"), dict) else sd

    def clean(k):
        if k.startswith("detector."):
            k = k.replace("detector.", "")
        return k.replace("student_trunk.", "")

    names = {clean(k) for k in sd}
    msd = model.state_dict()
    return sum(v.numel() for k, v in msd.items() if k in names) / max(sum(v.numel() for v in msd.values()), 1)


def load_student(st):
    """As L4 run 2 and the repo README's Quick Start; the processor keeps every detection above FLOOR."""
    ckpt = hf_hub_download(repo_id=HF_REPO, filename=st["rfilename"], revision=HF_SHA, token=os.environ.get("HF_TOKEN"))
    model = build_efficientsam3_image_model(
        checkpoint_path=ckpt, backbone_type=st["backbone_type"], model_name=st["model_name"],
        text_encoder_type=TEXT_ENCODER, text_encoder_context_length=TEXT_CONTEXT, load_from_HF=False, device=DEVICE)
    model.to(DEVICE).eval()
    st["coverage"] = round(checkpoint_coverage(model, ckpt), 4)
    assert st["coverage"] >= 0.95, f"{st['label']}: the checkpoint fills only {st['coverage']:.1%} of the model"
    return model, Sam3Processor(model, device=DEVICE, confidence_threshold=FLOOR)


print(f"✓ {len(STUDENTS)} models, {len(SAM3_DF)} photos; SAM 3 (E3, whole photos, 0.5) on them: Building IoU {SAM3_IOU:.4f}")

✓ 3 models, 21 photos; SAM 3 (E3, whole photos, 0.5) on them: Building IoU 0.2812


In [3]:
# ── Cell 3: The 21 before-photos and their true masks (the D4 rule: intact, then damaged, then destroyed) ──
FOLDER_TO_CLASS = {"train_building_intact_mask": 1, "train_building_damaged_mask": 2, "train_building_destroyed_mask": 3}


def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
              DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / filename):
        if c.exists():
            return str(c)
    return None


with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)
WANT = set(SAM3_DF["image"])
found = {}
for e in raw:
    if (e.get("task") != "Referring Expression Segmentation" or e.get("cls_description") != "Building Damage Assessment"
            or e.get("image_type") != "Optical"):
        continue
    name = Path(e.get("post_image_path", "").replace("\\", "/")).name
    if name not in WANT:
        continue
    d = found.setdefault(name, {"pre": e.get("pre_image_path"), "masks": {}})
    folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name
    m = resolve_path(e.get("ground_truth", ""))
    if m and folder in FOLDER_TO_CLASS:
        d["masks"][folder] = m
del raw
assert set(found) == WANT, f"not in the manifest: {sorted(WANT - set(found))}"

PHOTOS = []
for name in SAM3_DF["image"]:
    d = found[name]
    pre = cv2.cvtColor(cv2.imread(resolve_path(d["pre"])), cv2.COLOR_BGR2RGB)
    truth = None
    for folder, cls in FOLDER_TO_CLASS.items():
        if folder in d["masks"]:
            m = cv2.imread(d["masks"][folder], cv2.IMREAD_GRAYSCALE)
            truth = np.zeros_like(m, np.uint8) if truth is None else truth
            truth[m > 0] = cls
    if truth.shape != pre.shape[:2]:
        truth = cv2.resize(truth, (pre.shape[1], pre.shape[0]), interpolation=cv2.INTER_NEAREST)
    PHOTOS.append({"image": name, "pre": pre, "building": truth > 0})

# Check 1: the same photos and masks as E3
mine = np.array([int(p["building"].sum()) for p in PHOTOS])
e3_px = SAM3_DF["gt_px"].to_numpy()
assert abs(mine.sum() - e3_px.sum()) <= 0.005 * e3_px.sum(), f"true building px {mine.sum():,} against E3's {e3_px.sum():,}"
print(f"✓ Check 1: 21 photos, {mine.sum():,} true building px against E3's {e3_px.sum():,}; "
      f"{int((mine == e3_px).sum())} of 21 photos equal pixel for pixel")

✓ Check 1: 21 photos, 3,196,396 true building px against E3's 3,196,396; 21 of 21 photos equal pixel for pixel


In [4]:
# ── Cell 4: Check A, the repo's own example: each model finds the dog and the person? ──
def detect(processor, image_rgb, prompt):
    """Every detection above FLOOR, best first: (scores, boolean masks at the photo's size)."""
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        state = processor.set_image(Image.fromarray(image_rgb))
        out = processor.set_text_prompt(prompt=prompt, state=state)
    masks = out["masks"]
    masks = (masks[:, 0] if masks.ndim == 4 else masks).bool().cpu().numpy()
    scores = out["scores"].float().cpu().numpy()
    order = np.argsort(-scores)
    return scores[order], masks[order]


def overlay(image_rgb, masks, path):
    """The photo with the masks shaded magenta and outlined yellow."""
    out = image_rgb.copy()
    if len(masks):
        anym = np.any(masks, axis=0)
        out[anym] = (0.5 * out[anym] + 0.5 * np.array([255, 0, 255])).astype(np.uint8)
        for m in masks:
            contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out, contours, -1, (255, 255, 0), 2)
    cv2.imwrite(str(path), cv2.cvtColor(out, cv2.COLOR_RGB2BGR))


example_path = ESAM3_PKG_ROOT / "assets" / EXAMPLE[0]
example = cv2.cvtColor(cv2.imread(str(example_path)), cv2.COLOR_BGR2RGB)
MODELS, rows = {}, []
for st in STUDENTS:
    t0 = time.time()
    MODELS[st["label"]] = load_student(st)
    print(f"✓ {st['label']} loaded in {time.time() - t0:.1f} s; the checkpoint fills {st['coverage']:.1%} of the model")
    for prompt in EXAMPLE[1]:
        scores, masks = detect(MODELS[st["label"]][1], example, prompt)
        keep = scores >= 0.5
        rows.append({"model": st["label"], "prompt": prompt, "found_at_0.5": int(keep.sum()),
                     "top_score": round(float(scores[0]), 3) if len(scores) else 0.0})
        overlay(example, masks[keep], OUT_DIR / "overlays" / f"example_{st['label']}_{prompt}.png")
CHECK_A = pd.DataFrame(rows)
A_PASSES = bool((CHECK_A["found_at_0.5"] >= 1).all())
print(CHECK_A.to_string(index=False))
print(f"{'✓' if A_PASSES else '✗'} Check A: every model finds the dog and the person at 0.5: {A_PASSES}")

efficientsam3_ft/efficientsam3_efficient(…): reconstructing file:   0%|          |  0.00B /  468MB            

efficientsam3_ft/efficientsam3_efficient(…): downloading bytes:           |  0.00B            

Resizing positional embeddings from 77 to 16
✓ EV-M loaded in 9.3 s; the checkpoint fills 100.0% of the model


efficientsam3_ft/efficientsam3_repvit.pt: reconstructing file:   0%|          |  0.00B /  482MB            

efficientsam3_ft/efficientsam3_repvit.pt: downloading bytes:           |  0.00B            

Resizing positional embeddings from 77 to 16
✓ RV-M loaded in 11.3 s; the checkpoint fills 100.0% of the model


efficientsam3_ft/efficientsam3_tinyvit.p(…): reconstructing file:   0%|          |  0.00B /  493MB            

efficientsam3_ft/efficientsam3_tinyvit.p(…): downloading bytes:           |  0.00B            

Resizing positional embeddings from 77 to 16
✓ TV-M loaded in 8.2 s; the checkpoint fills 100.0% of the model
model prompt  found_at_0.5  top_score
 EV-M    dog             1      0.607
 EV-M person             2      0.872
 RV-M    dog             0      0.325
 RV-M person             2      0.822
 TV-M    dog             1      0.730
 TV-M person             2      0.851
✗ Check A: every model finds the dog and the person at 0.5: False


In [5]:
# ── Cell 5: Check B, the 21 satellite before-photos, whole, at four thresholds and three prompts ──
rows = []
t0 = time.time()
for st in STUDENTS:
    processor = MODELS[st["label"]][1]
    for prompt in PROMPTS:
        for ph in PHOTOS:
            scores, masks = detect(processor, ph["pre"], prompt)
            big = masks.reshape(len(masks), -1).sum(1) >= MIN_INSTANCE_PX if len(masks) else np.zeros(0, bool)
            row = {"model": st["label"], "prompt": prompt, "image": ph["image"],
                   "top_score": round(float(scores[0]), 4) if len(scores) else 0.0}
            for t in THRESHOLDS:
                sel = (scores >= t) & big
                found = np.any(masks[sel], axis=0) if sel.any() else np.zeros(ph["building"].shape, bool)
                row[f"n_{t}"] = int(sel.sum())
                row[f"inter_{t}"] = int((found & ph["building"]).sum())
                row[f"union_{t}"] = int((found | ph["building"]).sum())
            rows.append(row)
            if prompt == "building" and ph["image"] in OVERLAY_PHOTOS:
                sel = (scores >= THRESHOLDS[-1]) & big
                overlay(ph["pre"], masks[sel], OUT_DIR / "overlays" / f"{Path(ph['image']).stem}_{st['label']}_building_t{THRESHOLDS[-1]}.png")
PER_PHOTO = pd.DataFrame(rows)

summary = []
for (m, p), d in PER_PHOTO.groupby(["model", "prompt"], sort=False):
    r = {"model": m, "prompt": p, "median_top_score": round(float(d["top_score"].median()), 3),
         "max_top_score": round(float(d["top_score"].max()), 3)}
    for t in THRESHOLDS:
        r[f"building_iou_{t}"] = round(d[f"inter_{t}"].sum() / max(d[f"union_{t}"].sum(), 1), 4)
        r[f"detections_{t}"] = int(d[f"n_{t}"].sum())
    summary.append(r)
SUMMARY = pd.DataFrame(summary)
print(f"✓ {len(PER_PHOTO)} model-prompt-photo runs in {(time.time() - t0) / 60:.1f} min")
print(SUMMARY.to_string(index=False))
print(f"Reference: SAM 3 on the same photos (E3, whole, 0.5): Building IoU {SAM3_IOU:.4f}, "
      f"{int(SAM3_DF['instances'].sum())} detections, {int(SAM3_DF['gt_blobs'].sum())} true buildings")

✓ 189 model-prompt-photo runs in 1.1 min
model   prompt  median_top_score  max_top_score  building_iou_0.5  detections_0.5  building_iou_0.3  detections_0.3  building_iou_0.1  detections_0.1  building_iou_0.05  detections_0.05
 EV-M building             0.051          0.760            0.0023               2            0.0456              15            0.0605             227             0.2197              652
 EV-M    house             0.055          0.411            0.0000               0            0.0035               4            0.0463             203             0.1059              615
 EV-M     roof             0.145          0.501            0.0000               1            0.0017              10            0.0638             268             0.1283             1020
 RV-M building             0.254          0.578            0.0139               5            0.1217              38            0.3195             761             0.3000             1599
 RV-M    house             0.

In [6]:
# ── Cell 6: The reading fixed before the run, and the files ──
iou_cols = [f"building_iou_{t}" for t in THRESHOLDS]
bld = SUMMARY[SUMMARY["prompt"] == "building"].set_index("model")[iou_cols]
best_model, best_col = bld.stack().idxmax()
BEST = float(bld.loc[best_model, best_col])
others = SUMMARY[SUMMARY["prompt"] != "building"]
best_other = others.loc[others[iou_cols].max(axis=1).idxmax()]

if not A_PASSES:
    READING = "setup"
    verdict = ("Check A fails: a model finds no dog or no person in the repo's own example at 0.5, so our setup is "
               "broken and L4's result says nothing about EfficientSAM3 until it is fixed.")
elif BEST >= 0.20:
    READING = "calibration"
    verdict = (f"The models find buildings but score them low: {best_model} reaches Building IoU {BEST:.4f} with "
               f"\"building\" at {best_col.split('_')[-1]} (SAM 3 here: {SAM3_IOU:.4f}). A new run on the 605 clean "
               "pairs at that model and threshold becomes a candidate, with its own rule.")
elif BEST < 0.10:
    READING = "domain"
    verdict = (f"Check A passes, and with \"building\" no model reaches 0.10 at any threshold (best: {best_model}, "
               f"{BEST:.4f} at {best_col.split('_')[-1]}; SAM 3 here: {SAM3_IOU:.4f}). The models don't see our "
               "buildings from above: L4 stands as measured.")
else:
    READING = "partly"
    verdict = (f"Check A passes; with \"building\" the best is {best_model} at {BEST:.4f} "
               f"({best_col.split('_')[-1]}), between 0.10 and 0.20. Reported; nothing decided.")
print(f"Reading: {READING}. {verdict}")
print(f"Beside it, deciding nothing: the best other prompt is \"{best_other['prompt']}\" with {best_other['model']}, "
      f"Building IoU {best_other[iou_cols].max():.4f}.")


def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"


CHECK_A.to_csv(OUT_DIR / "check_a_example.csv", index=False)
PER_PHOTO.to_csv(OUT_DIR / "check_b_per_photo.csv", index=False)
SUMMARY.to_csv(OUT_DIR / "check_b_summary.csv", index=False)
config = {"task": "L4b", "efficientsam3_commit": ESAM3_COMMIT, "hf_weights": f"{HF_REPO}@{HF_SHA}",
          "text_encoder": TEXT_ENCODER, "text_context": TEXT_CONTEXT, "floor": FLOOR, "thresholds": THRESHOLDS,
          "prompts": PROMPTS, "photos": SAM3_DF["image"].tolist(), "sam3_reference_iou": round(float(SAM3_IOU), 4),
          "coverage": {st["label"]: st["coverage"] for st in STUDENTS}, "check_a_passes": A_PASSES,
          "reading": READING, "verdict": verdict, "torch": torch.__version__}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# L4b: is L4's near-zero result EfficientSAM3 or our setup?", "",
    f"EfficientSAM3 `{ESAM3_COMMIT[:7]}`, weights `{HF_REPO}@{HF_SHA[:8]}`, {TEXT_ENCODER} at {TEXT_CONTEXT} tokens, "
    "half precision on a T4. The question, the reading and the expectation are in the notebook's first cell.", "",
    "## Check A: the repo's own example (`dog_person.jpeg`, threshold 0.5)", "", md(CHECK_A), "",
    f"Every model finds the dog and the person: **{A_PASSES}**. Overlays: `overlays/example_*.png`.", "",
    "## Check B: 21 satellite before-photos, whole", "",
    f"Building IoU over the 21 photos at each threshold, the detections kept and the top scores. SAM 3 on the same "
    f"photos (E3, whole, 0.5): **{SAM3_IOU:.4f}**.", "", md(SUMMARY), "",
    "## Reading", "", f"**{READING}.** {verdict}", "",
    f"Beside it, deciding nothing: the best other prompt is \"{best_other['prompt']}\" with {best_other['model']}, "
    f"Building IoU {best_other[iou_cols].max():.4f}.", "",
    "Files: `check_a_example.csv`, `check_b_per_photo.csv`, `check_b_summary.csv`, `config.json`, `overlays/`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote README.md, config.json, three CSVs and {len(list((OUT_DIR / 'overlays').glob('*.png')))} overlays to {OUT_DIR}")

Reading: setup. Check A fails: a model finds no dog or no person in the repo's own example at 0.5, so our setup is broken and L4's result says nothing about EfficientSAM3 until it is fixed.
Beside it, deciding nothing: the best other prompt is "house" with RV-M, Building IoU 0.3382.
✓ Wrote README.md, config.json, three CSVs and 12 overlays to /kaggle/working/L4b
